# Applying Custom Functions with `.apply()`

While `.map()` is great for simple dictionary translation, sometimes you need to apply actual **programming logic** (such as `if/else` conditions or mathematical formulas) to your data. This is where `.apply()` shines.

With `.apply()`, you can:
1.  Apply a function **element-wise** to a single column (Series).
2.  Apply a function **row-wise** across multiple columns of a DataFrame by setting `axis=1`.

Row-wise application is extremely useful when your calculation depends on multiple variables (for example, calculating Body Mass Index (BMI) using both `Weight` and `Height` columns).

### Plain English Explanation & Real-World Analogy
Imagine you are a **manufacturing inspector** standing at a conveyor belt of products:
*   **Column-wise `.apply()`**: You look at one specific property (like weight). For every product, you weigh it and write down its status (e.g., *"Pass"* if over 100g, *"Fail"* if under). You do this one by one down the column.
*   **Row-wise `.apply(axis=1)`**: You look at the entire product as a whole. You read its label, inspect its packaging, and check its weight *together* to determine its overall quality category. You are evaluating across multiple properties for each single row.

### Code Examples

Let's create a DataFrame of athletes with their heights and weights:


In [1]:
import pandas as pd

athletes = pd.DataFrame({
    'Name': ['Michael', 'Sophia', 'David', 'Emma'],
    'Height_cm': [195, 160, 180, 150],
    'Weight_kg': [95, 52, 85, 45]
})
print(athletes)

      Name  Height_cm  Weight_kg
0  Michael        195         95
1   Sophia        160         52
2    David        180         85
3     Emma        150         45


#### Column-wise `.apply()` with a `lambda` Function
Let's classify athletes into "Tall" or "Short" based on their height. If they are over 175 cm, they are Tall; otherwise, they are Short:

In [2]:
# Apply an inline lambda function to Height_cm
athletes['Height_Class'] = athletes['Height_cm'].apply(lambda x: 'Tall' if x > 175 else 'Short')
print(athletes[['Name', 'Height_cm', 'Height_Class']])

      Name  Height_cm Height_Class
0  Michael        195         Tall
1   Sophia        160        Short
2    David        180         Tall
3     Emma        150        Short


#### Row-wise `.apply(axis=1)` with a Custom Python Function
Let's write a standard Python function that takes an entire row of data, evaluates both height and weight, and returns an athletic division category:

In [3]:
# Define custom function that accepts a row (Series)
def categorize_athlete(row):
    height = row['Height_cm']
    weight = row['Weight_kg']

    if height > 185 and weight > 90:
        return 'Heavyweight Giant'
    elif height < 165 and weight < 55:
        return 'Lightweight Compact'
    else:
        return 'Standard Athlete'

# Apply row-wise across the DataFrame (axis=1 is crucial!)
athletes['Division'] = athletes.apply(categorize_athlete, axis=1)
print(athletes)

      Name  Height_cm  Weight_kg Height_Class             Division
0  Michael        195         95         Tall    Heavyweight Giant
1   Sophia        160         52        Short  Lightweight Compact
2    David        180         85         Tall     Standard Athlete
3     Emma        150         45        Short  Lightweight Compact


### Common Mistakes Beginners Make
1.  **Forgetting `axis=1` in row-wise application**: If you want to run a custom function that compares multiple columns, you *must* specify `axis=1` (columns). If you omit it, Pandas will default to `axis=0` (rows), trying to pass entire columns to your function, which will cause a `KeyError`.
2.  **Using `.apply()` when a vectorized method exists**: Python custom functions run in standard Python loops under the hood, which is slower than optimized NumPy C-compiled operations. If you just want to multiply two columns, write `df['A'] * df['B']` (vectorized) instead of using `.apply()`. Use `.apply()` only when your logic requires complex `if/else` structures.


#### Exercise 1 (Medium)
Given a DataFrame of employee salaries and performance ratings:
```python
employees = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie'],
    'Salary': [80000, 100000, 120000],
    'Rating': ['Excellent', 'Average', 'Excellent']
})
```
Write a custom Python function and use `.apply(..., axis=1)` to calculate their final bonus:
*   If their rating is `'Excellent'`, they get a `10%` bonus of their salary.
*   Otherwise, they get a `2%` bonus of their salary.
Store the bonus in a new column called `'Bonus'`.

In [4]:
import pandas as pd

employees = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie'],
    'Salary': [80000, 100000, 120000],
    'Rating': ['Excellent', 'Average', 'Excellent']
})

# Define the custom function for rows
def calculate_bonus(row):
    salary = row['Salary']
    rating = row['Rating']

    if rating == 'Excellent':
        return salary * 0.10
    else:
        return salary * 0.02

# Apply row-wise
employees['Bonus'] = employees.apply(calculate_bonus, axis=1)
print(employees)

      Name  Salary     Rating    Bonus
0    Alice   80000  Excellent   8000.0
1      Bob  100000    Average   2000.0
2  Charlie  120000  Excellent  12000.0
